In [0]:
# ================================================================
# ADF Dynamic Medallion Pipeline — Configuration
# ================================================================
# Problem: ADF ingests data, but there is NO raw/stg ADLS container.
# Solution: Use a UC Volume as the landing zone + a metadata-driven
#           control table to build Bronze → Silver → Gold dynamically.
# ================================================================

from pyspark.sql import functions as F

# ── Unity Catalog & Schema Configuration ──
CATALOG_NAME    = "adf_catalog"
BRONZE_SCHEMA   = "bronze"
SILVER_SCHEMA   = "silver"
GOLD_SCHEMA     = "gold"
CONTROL_SCHEMA  = "control"

# ── UC Volume (replaces the raw/stg ADLS container) ──
VOLUME_NAME = "landing"
VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{CONTROL_SCHEMA}/{VOLUME_NAME}"

# ── Control / Metadata Table ──
CONTROL_TABLE = f"{CATALOG_NAME}.{CONTROL_SCHEMA}.pipeline_metadata"

print(f"Catalog:      {CATALOG_NAME}")
print(f"Bronze:       {CATALOG_NAME}.{BRONZE_SCHEMA}")
print(f"Silver:       {CATALOG_NAME}.{SILVER_SCHEMA}")
print(f"Gold:         {CATALOG_NAME}.{GOLD_SCHEMA}")
print(f"Volume:       {VOLUME_PATH}")
print(f"Control Tbl:  {CONTROL_TABLE}")

In [0]:
# ================================================================
# Create UC Catalog, Schemas, and Landing Volume
# ================================================================
# The UC Volume replaces the traditional raw/stg ADLS container.
# ADF writes Parquet files here; Databricks reads them into Bronze.
# ================================================================

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
spark.sql(f"USE CATALOG {CATALOG_NAME}")

for schema in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA, CONTROL_SCHEMA]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{schema}")
    print(f"  ✓ Schema: {CATALOG_NAME}.{schema}")

# Landing volume — ADF's target sink (no ADLS container needed)
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{CONTROL_SCHEMA}.{VOLUME_NAME}")
print(f"  ✓ Volume: {VOLUME_PATH}")
print("\nAll infrastructure created.")

In [0]:
# ================================================================
# Create the Control / Metadata Table
# ================================================================
# This table drives the ENTIRE pipeline. Each row = one source table.
# To add a new table: INSERT one row → pipeline picks it up automatically.
# No code changes needed — this is what makes it "dynamic."
# ================================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CONTROL_TABLE} (
    source_table_name     STRING   NOT NULL,
    bronze_table_name     STRING   NOT NULL,
    silver_table_name     STRING   NOT NULL,
    gold_table_name       STRING,
    primary_keys          STRING   NOT NULL,
    volume_subpath        STRING   NOT NULL,
    silver_transform_sql  STRING,
    gold_transform_sql    STRING,
    is_active             BOOLEAN  NOT NULL
)
""")

# Clear and insert metadata for 3 sample source tables
spark.sql(f"TRUNCATE TABLE {CONTROL_TABLE}")

spark.sql(f"""
INSERT INTO {CONTROL_TABLE} VALUES
-- ── Table 1: Customers ──
(
  'customers',
  '{CATALOG_NAME}.{BRONZE_SCHEMA}.customers',
  '{CATALOG_NAME}.{SILVER_SCHEMA}.customers',
  '{CATALOG_NAME}.{GOLD_SCHEMA}.customers_summary',
  'customer_id',
  'customers',
  'SELECT
     customer_id,
     UPPER(TRIM(customer_name)) AS customer_name,
     LOWER(TRIM(email)) AS email,
     TRIM(city) AS city,
     TRIM(country) AS country,
     CAST(signup_date AS DATE) AS signup_date,
     _ingested_at,
     _source_file
   FROM {CATALOG_NAME}.{BRONZE_SCHEMA}.customers',
  'SELECT
     country,
     COUNT(*) AS total_customers,
     MIN(signup_date) AS earliest_signup,
     MAX(signup_date) AS latest_signup
   FROM {CATALOG_NAME}.{SILVER_SCHEMA}.customers
   GROUP BY country',
  true
),
-- ── Table 2: Orders ──
(
  'orders',
  '{CATALOG_NAME}.{BRONZE_SCHEMA}.orders',
  '{CATALOG_NAME}.{SILVER_SCHEMA}.orders',
  '{CATALOG_NAME}.{GOLD_SCHEMA}.orders_summary',
  'order_id',
  'orders',
  'SELECT
     order_id,
     customer_id,
     product_id,
     CAST(order_date AS DATE) AS order_date,
     CAST(quantity AS INT) AS quantity,
     CAST(amount AS DOUBLE) AS amount,
     UPPER(TRIM(status)) AS status,
     _ingested_at,
     _source_file
   FROM {CATALOG_NAME}.{BRONZE_SCHEMA}.orders',
  'SELECT
     DATE_TRUNC(''month'', order_date) AS order_month,
     COUNT(*) AS total_orders,
     SUM(quantity) AS total_quantity,
     SUM(amount) AS total_revenue
   FROM {CATALOG_NAME}.{SILVER_SCHEMA}.orders
   GROUP BY DATE_TRUNC(''month'', order_date)',
  true
),
-- ── Table 3: Products (no Gold table — gold_table_name = NULL) ──
(
  'products',
  '{CATALOG_NAME}.{BRONZE_SCHEMA}.products',
  '{CATALOG_NAME}.{SILVER_SCHEMA}.products',
  NULL,
  'product_id',
  'products',
  'SELECT
     product_id,
     UPPER(TRIM(product_name)) AS product_name,
     TRIM(category) AS category,
     CAST(price AS DOUBLE) AS price,
     _ingested_at,
     _source_file
   FROM {CATALOG_NAME}.{BRONZE_SCHEMA}.products',
  NULL,
  true
)
""")

print(f"Control table created: {CONTROL_TABLE}")
display(spark.sql(f"""
  SELECT source_table_name, bronze_table_name, silver_table_name, gold_table_name, is_active
  FROM {CONTROL_TABLE}
  ORDER BY source_table_name
"""))

In [0]:
# ================================================================
# Simulate ADF Landing — Write Parquet to UC Volume
# ================================================================
# In PRODUCTION: ADF Copy Activity writes Parquet files to this volume.
#   Source: On-prem SQL Server (via Self-Hosted IR)
#   Sink:   /Volumes/adf_catalog/landing/<table>/  (Parquet format)
# Here we SIMULATE that by generating sample data with intentional
# dirtiness (whitespace, string numbers) to show Silver cleansing.
# ================================================================

from pyspark.sql import Row

# ── Simulate "customers" landing ──
customers_data = [
    Row(customer_id=1, customer_name="  john doe  ", email="  john@example.com ", city="New York",   country="USA",     signup_date="2023-01-15"),
    Row(customer_id=2, customer_name="jane smith",   email="jane@example.com",   city="London",     country="UK",      signup_date="2023-03-22"),
    Row(customer_id=3, customer_name="  bob lee  ",   email="bob@example.com",    city="Tokyo",      country="Japan",   signup_date="2023-06-10"),
    Row(customer_id=4, customer_name="alice brown",   email="  alice@example.com  ", city="Berlin",  country="Germany", signup_date="2023-08-05"),
]
spark.createDataFrame(customers_data) \
    .write.mode("overwrite") \
    .parquet(f"{VOLUME_PATH}/customers/")
print("  ✓ Landed: customers (4 rows)")

# ── Simulate "orders" landing ──
orders_data = [
    Row(order_id=101, customer_id=1, product_id=501, order_date="2024-01-10", quantity=3,  amount="150.50", status="  shipped  "),
    Row(order_id=102, customer_id=2, product_id=502, order_date="2024-02-15", quantity=1,  amount="49.99",  status="pending"),
    Row(order_id=103, customer_id=1, product_id=503, order_date="2024-03-20", quantity=5,  amount="299.95", status="delivered"),
    Row(order_id=104, customer_id=3, product_id=501, order_date="2024-04-01", quantity=2,  amount="100.00", status="shipped"),
    Row(order_id=105, customer_id=4, product_id=502, order_date="2024-05-12", quantity=10, amount="499.90", status="  cancelled  "),
]
spark.createDataFrame(orders_data) \
    .write.mode("overwrite") \
    .parquet(f"{VOLUME_PATH}/orders/")
print("  ✓ Landed: orders (5 rows)")

# ── Simulate "products" landing ──
products_data = [
    Row(product_id=501, product_name="  widget pro  ", category="  electronics ", price="19.99"),
    Row(product_id=502, product_name="gadget lite",     category="accessories",   price="49.99"),
    Row(product_id=503, product_name="super tool",      category="  tools  ",      price="59.99"),
]
spark.createDataFrame(products_data) \
    .write.mode("overwrite") \
    .parquet(f"{VOLUME_PATH}/products/")
print("  ✓ Landed: products (3 rows)")

# ── List what ADF "landed" ──
print(f"\nFiles in landing volume ({VOLUME_PATH}):")
for entry in dbutils.fs.ls(VOLUME_PATH):
    print(f"  📁 {entry.name}")
    for sub_entry in dbutils.fs.ls(entry.path):
        print(f"      📄 {sub_entry.name}")

In [0]:
# ================================================================
# Dynamic Bronze Layer
# ================================================================
# Loop over the control table → create a Bronze Delta table for each
# active source table by reading Parquet from the UC Volume.
# Adds _ingested_at and _source_file metadata columns.
# This replaces traditional "COPY INTO from raw ADLS container."
# ================================================================

control_rows = spark.sql(f"""
    SELECT source_table_name, bronze_table_name, volume_subpath
    FROM {CONTROL_TABLE}
    WHERE is_active = true
    ORDER BY source_table_name
""").collect()

print(f"Building Bronze layer for {len(control_rows)} tables...\n")

for row in control_rows:
    source_table = row['source_table_name']
    bronze_table = row['bronze_table_name']
    volume_subpath = row['volume_subpath']
    file_path = f"{VOLUME_PATH}/{volume_subpath}"

    # Read Parquet from UC Volume → add metadata → write Bronze Delta table
    df = (spark.read
          .parquet(f"{file_path}/")
          .withColumn("_ingested_at",  F.current_timestamp())
          .withColumn("_source_file", F.col("_metadata.file_path")))

    df.write.mode("overwrite").saveAsTable(bronze_table)

    count = spark.sql(f"SELECT COUNT(*) FROM {bronze_table}").collect()[0][0]
    print(f"  ✓ Bronze: {bronze_table} ({count} rows)")

print("\n✅ Bronze layer complete.")

In [0]:
# ================================================================
# Dynamic Silver Layer
# ================================================================
# Loop over the control table → create a Silver Delta table for each
# active source table by executing the silver_transform_sql (a SQL
# SELECT from Bronze with cleansing: TRIM, UPPER, CAST, etc.).
# In production, use MERGE INTO with primary_keys for incremental loads.
# ================================================================

control_rows = spark.sql(f"""
    SELECT source_table_name, bronze_table_name, silver_table_name, primary_keys, silver_transform_sql
    FROM {CONTROL_TABLE}
    WHERE is_active = true
    ORDER BY source_table_name
""").collect()

print(f"Building Silver layer for {len(control_rows)} tables...\n")

for row in control_rows:
    source_table  = row['source_table_name']
    bronze_table  = row['bronze_table_name']
    silver_table  = row['silver_table_name']
    primary_keys  = row['primary_keys']
    transform_sql = row['silver_transform_sql']

    # Execute the cleansing SQL (SELECT from Bronze → cleansed Silver)
    silver_df = spark.sql(transform_sql)
    silver_df.write.mode("overwrite").saveAsTable(silver_table)

    count = spark.sql(f"SELECT COUNT(*) FROM {silver_table}").collect()[0][0]
    print(f"  ✓ Silver: {silver_table} ({count} rows)")

print("\n✅ Silver layer complete.")

# Show a Silver sample to demonstrate cleansing
print("\n── Silver Sample: customers (notice cleaned names/emails) ──")
display(spark.sql(f"SELECT customer_id, customer_name, email, city, country, signup_date FROM {CATALOG_NAME}.{SILVER_SCHEMA}.customers"))

In [0]:
# ================================================================
# Dynamic Gold Layer
# ================================================================
# Loop over the control table → create a Gold Delta table for each
# active source table where gold_table_name IS NOT NULL by executing
# the gold_transform_sql (aggregation SQL from Silver).
# ================================================================

control_rows = spark.sql(f"""
    SELECT source_table_name, gold_table_name, gold_transform_sql
    FROM {CONTROL_TABLE}
    WHERE is_active = true AND gold_table_name IS NOT NULL
    ORDER BY source_table_name
""").collect()

print(f"Building Gold layer for {len(control_rows)} tables...\n")

for row in control_rows:
    source_table  = row['source_table_name']
    gold_table    = row['gold_table_name']
    transform_sql = row['gold_transform_sql']

    # Execute the aggregation SQL (SELECT from Silver → Gold)
    gold_df = spark.sql(transform_sql)
    gold_df.write.mode("overwrite").saveAsTable(gold_table)

    count = spark.sql(f"SELECT COUNT(*) FROM {gold_table}").collect()[0][0]
    print(f"  ✓ Gold: {gold_table} ({count} rows)")

print("\n✅ Gold layer complete.")

In [0]:
# ================================================================
# Validation — Row Counts & Sample Data for All Layers
# ================================================================

print("=" * 75)
print("  MEDALLION ARCHITECTURE — VALIDATION SUMMARY")
print("=" * 75)

all_tables = spark.sql(f"""
    SELECT source_table_name, bronze_table_name, silver_table_name, gold_table_name
    FROM {CONTROL_TABLE}
    WHERE is_active = true
    ORDER BY source_table_name
""").collect()

for row in all_tables:
    src = row['source_table_name']
    print(f"\n  ─── {src.upper()} ───")

    for layer_name, table_name in [
        ("Bronze", row['bronze_table_name']),
        ("Silver", row['silver_table_name']),
        ("Gold",   row['gold_table_name']),
    ]:
        if table_name:
            count = spark.sql(f"SELECT COUNT(*) FROM {table_name}").collect()[0][0]
            print(f"    {layer_name:8s} {table_name:55s} {count:>5} rows")

# ── Show Gold layer samples ──
print("\n" + "=" * 75)
print("  GOLD SAMPLE: customers_summary (by country)")
print("=" * 75)
display(spark.sql(f"SELECT * FROM {CATALOG_NAME}.{GOLD_SCHEMA}.customers_summary ORDER BY country"))

print("\n" + "=" * 75)
print("  GOLD SAMPLE: orders_summary (by month)")
print("=" * 75)
display(spark.sql(f"SELECT * FROM {CATALOG_NAME}.{GOLD_SCHEMA}.orders_summary ORDER BY order_month"))

# ── Final message ──
print("\n" + "=" * 75)
print("  ✅ Pipeline complete: ADF → UC Volume → Bronze → Silver → Gold")
print("  No raw/stg ADLS container needed — UC Volume serves as landing zone.")
print("  Dynamic: add a table by inserting one row into the control table.")
print("=" * 75)